# Stage 5 Final Inference Notebook

Этот ноутбук нужен для использования уже обученной модели.

Что умеет ноутбук:
- подключает Google Drive
- загружает веса модели и class mapping
- предобрабатывает изображение
- делает предсказание
- показывает понятный результат

Для обычной проверки достаточно запустить ноутбук сверху вниз и выполнить последнюю тестовую ячейку.


In [ ]:
!python -m pip install -q Pillow torch torchvision matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

PACKAGE_DIR = Path('/content/drive/MyDrive/MyProject1/stage5_package')
MODEL_PATH = PACKAGE_DIR / 'resnet34_best_model.pth'
CLASS_MAPPING_PATH = PACKAGE_DIR / 'class_mapping.json'
DEFAULT_IMAGE_PATH = PACKAGE_DIR / 'sample_images' / 'punching_hole_sample.jpg'
IMAGE_SIZE = 160
MODEL_NAME = 'resnet34'
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

print('PACKAGE_DIR        =', PACKAGE_DIR)
print('MODEL_PATH         =', MODEL_PATH)
print('CLASS_MAPPING_PATH =', CLASS_MAPPING_PATH)
print('DEFAULT_IMAGE_PATH =', DEFAULT_IMAGE_PATH)
print('MODEL_NAME         =', MODEL_NAME)
print('DEVICE             =', DEVICE)


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from torchvision.models import ResNet34_Weights, resnet34


def load_class_mapping(class_mapping_path):
    with open(class_mapping_path, 'r', encoding='utf-8') as fp:
        raw_mapping = json.load(fp)
    ordered_targets = sorted(raw_mapping.keys(), key=int)
    return {int(target): raw_mapping[target]['class_name'] for target in ordered_targets}


def build_preprocess(image_size=160):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def load_image(image_path):
    return Image.open(image_path).convert('RGB')


def preprocess_image(image, preprocess):
    return preprocess(image).unsqueeze(0)


def build_model(num_classes, pretrained=False, model_name='resnet34'):
    if model_name != 'resnet34':
        raise ValueError(f'Unsupported model: {model_name}')
    weights = ResNet34_Weights.DEFAULT if pretrained else None
    model = resnet34(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def load_trained_model(model_path, num_classes, device='cpu', model_name='resnet34'):
    model = build_model(num_classes=num_classes, pretrained=False, model_name=model_name)
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model


def predict_image(image_path, model, class_mapping, preprocess, device='cpu'):
    image = load_image(image_path)
    inputs = preprocess_image(image, preprocess).to(device)
    with torch.no_grad():
        logits = model(inputs)
        probabilities = torch.softmax(logits, dim=1)[0]
        pred_target = int(torch.argmax(probabilities).item())
        confidence = float(probabilities[pred_target].item())
    return {
        'image': image,
        'pred_target': pred_target,
        'pred_class_name': class_mapping[pred_target],
        'confidence': confidence,
    }


def show_prediction_result(result, image_path):
    plt.figure(figsize=(6, 6))
    plt.imshow(result['image'])
    plt.axis('off')
    plt.title(f"Prediction: {result['pred_class_name']}\nConfidence: {result['confidence']:.4f}")
    plt.show()
    print('Image path   :', image_path)
    print('Prediction   :', result['pred_class_name'])
    print('Confidence   :', round(result['confidence'], 4))


def run_inference(image_path=DEFAULT_IMAGE_PATH, model_path=MODEL_PATH, class_mapping_path=CLASS_MAPPING_PATH, image_size=IMAGE_SIZE, device=DEVICE, model_name=MODEL_NAME):
    image_path = Path(image_path)
    model_path = Path(model_path)
    class_mapping_path = Path(class_mapping_path)
    assert image_path.exists(), f'Image not found: {image_path}'
    assert model_path.exists(), f'Model not found: {model_path}'
    assert class_mapping_path.exists(), f'Class mapping not found: {class_mapping_path}'
    class_mapping = load_class_mapping(class_mapping_path)
    preprocess = build_preprocess(image_size=image_size)
    model = load_trained_model(model_path=model_path, num_classes=len(class_mapping), device=device, model_name=model_name)
    result = predict_image(image_path=image_path, model=model, class_mapping=class_mapping, preprocess=preprocess, device=device)
    show_prediction_result(result, image_path)
    return result


In [ ]:
sample_result = run_inference()
sample_result


In [ ]:
from google.colab import files

uploaded = files.upload()
if uploaded:
    uploaded_image_path = next(iter(uploaded.keys()))
    uploaded_result = run_inference(image_path=uploaded_image_path)
    uploaded_result
else:
    print('Файл не был загружен.')
